In [1]:
import numpy as np
import torch
from flow_matching import *
from dit import *

# --------------------------------------
# Load mel spec
# --------------------------------------
npz_path = "data/mel_specs/001.clar.npz"     # <-- change this
data = np.load(npz_path)
mel = data["mel"]                  # shape: (n_mels=128, T)

# Use exactly first 128 frames
mel = mel[:, :128]                 # (128, 128)


# --------------------------------------
# Prepare model input (B, L, D)
# --------------------------------------
mel = torch.from_numpy(mel).float()         # (128, 128)
mel = mel.unsqueeze(0)                      # (1, 128, 128)
cond_inst = mel.clone()                     # conditioning input

# --------------------------------------
# Instantiate model & flow wrapper
# --------------------------------------
input_dim = mel.shape[-1]   # e.g., 128

dit = DiT(
    input_dim=input_dim,
    num_heads=4,
    head_dim=input_dim // 4,
    n_blocks=4,
    cond_dim=input_dim,
)

flow = FlowModel(dit, cfg_drop_prob=0.1)
flow = flow.to(mel.device)

# --------------------------------------
# Run sampling
# --------------------------------------
print(cond_inst.shape)
with torch.no_grad():
    out = flow.sample(cond_inst, steps=32, cfg_strength=3.0)

print("Input shape: ", cond_inst.shape)
print("Output shape:", out.shape)

torch.Size([1, 128, 128])
Input shape:  torch.Size([1, 128, 128])
Output shape: torch.Size([1, 128, 128])
